In [ ]:
import numpy as np
from scipy import stats
import xarray as xr
import pandas as pd
import os
from netCDF4 import Dataset,num2date 
import time
import matplotlib.pyplot as plt
import cdsapi
import time
import pypsa
import seaborn as sns

# SLR to D-SLR

In [ ]:
# change paths to wherever you've downloaded the line_adj/line_discount csvs
path = "line_adj.csv"
slr_adj = pd.read_csv(path,header=0,index_col=0)
path = "line_discount.csv"
dslr = pd.read_csv(path,header=0,index_col=0)

In [ ]:
gcms = ["ec3","ec3veg","miroc6","mpi","taiesm1"]

In [ ]:
means = []
maxes = []
mins = []
for gcm in gcms:
    sa = slr_adj[gcm]
    ds = dslr[gcm]
    
    diff = (ds-sa)/sa*100
    print(f"Mean Shift: {diff.mean():.2f}%")
    print(f"Max Shift: {diff.max():.2f}%")
    print(f"Min Shift: {diff.min():.2f}%")
    means = np.append(means,diff.mean())
    maxes = np.append(maxes,diff.max())
    mins = np.append(mins,diff.min())
    
print(f"Climate impacts result in a {np.min(mins):.2f} to {np.max(maxes):.2f}% change in capacity, for an average static line capacity reduction of {np.average(means):.2f}% across GCMs.")

# AAR & DLR

In [ ]:
def calc_below(df):
    return (df < 1).sum().sum() / df.size * 100
def get_min(df):
    return df.min().mean()
def get_rel(df1,df2):
    return (df1 < df2).sum().sum() / df1.size * 100

In [ ]:
dlr_2025 = {}
dlr_2050 = {}
aar_2025 = {}
aar_2050 = {}

for gcm in gcms:
    start = time.time()
    print(f"Starting {gcm}")
    
    path = f"rel-dlr_2025_WUS-{gcm}_base-dc-line_phi20-70-div3.csv"
    dlr_2025[gcm] = pd.read_csv(path,header=0,index_col=0)
    print(f"Loaded DLR 2025 ({time.time() - start:.2f}s)")

    path = f"rel-aar_2025_WUS-{gcm}_base-dc-line_6.csv"
    aar_2025[gcm] = pd.read_csv(path,header=0,index_col=0)
    print(f"Loaded AAR 2025 ({time.time() - start:.2f}s)")

    path = f"rel-dlr_2050_WUS-{gcm}_base-dc-line_phi20-70-div3.csv"
    dlr_2050[gcm] = pd.read_csv(path,header=0,index_col=0)
    print(f"Loaded DLR 2050 ({time.time() - start:.2f}s)")

    path = f"rel-aar_2050_WUS-{gcm}_base-dc-line_6.csv"
    aar_2050[gcm] = pd.read_csv(path,header=0,index_col=0)
    print(f"Loaded AAR 2050 ({time.time() - start:.2f}s)")

In [ ]:
stats = pd.DataFrame(index=gcms,columns=["DLR mean shift","AAR mean shift","1st DLR shift","1st AAR shift"])
for gcm in gcms:
    start = time.time()
    print(f"Calculating stats for {gcm}")
    md2025 = dlr_2025[gcm].mean()
    md2050 = dlr_2050[gcm].mean()
    dmd = (md2050 - md2025)/md2025*100
    stats.loc[gcm,"DLR mean shift"] = dmd.mean()
    
    p1d2025 = dlr_2025[gcm].quantile(0.01)
    p1d2050 = dlr_2050[gcm].quantile(0.01)
    dp1d = (p1d2050 - p1d2025)/p1d2025*100
    stats.loc[gcm,"1st DLR shift"] = dp1d.mean()
    
    
    ma2025 = aar_2025[gcm].mean()
    ma2050 = aar_2050[gcm].mean()
    dma = (ma2050 - ma2025)/ma2025*100
    stats.loc[gcm,"AAR mean shift"] = dma.mean()
    
    p1a2025 = aar_2025[gcm].quantile(0.01)
    p1a2050 = aar_2050[gcm].quantile(0.01)
    dp1a = (p1a2050 - p1a2025)/p1a2025*100
    stats.loc[gcm,"1st AAR shift"] = dp1a.mean()
    (f"{gcm} finished ({time.time() - start:.2f}s)")
print(stats)   
print(stats.mean())